In [83]:
import numpy as np
import matplotlib.pyplot as plt
import math
import joblib
import torch
import torch.nn as nn
import torchvision
import deepRD.tools.trajectoryTools as trajectoryTools
from torch.utils.data import Dataset, DataLoader
from torch import optim
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler

In [50]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [71]:
class MLP(nn.Module):
    def __init__(self, in_dim, out_dim, hidden=(128,128)):
        super().__init__()
        layers, d = [], in_dim
        for h in hidden:
            layers += [nn.Linear(d, h), nn.SiLU(), nn.LayerNorm(h)]
            d = h
        layers += [nn.Linear(d, out_dim)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

def reparam(mu, logvar):
    eps = torch.randn_like(mu)
    return mu + eps * torch.exp(0.5 * logvar)

class DiagGaussianHead(nn.Module):
    """Outputs (mu, log_sigma) for R^3."""
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.mlp = MLP(in_dim, out_dim, hidden=(128,128))
    def forward(self, x):
        out = self.mlp(x)
        mu, log_sig = out[..., :3], out[..., 3:]
        return mu, log_sig

# ---------- CVAE ----------
class CVAE(nn.Module):
    def __init__(self, idim=3, cdim=6, zdim=3):
        super().__init__()
        self.cdim, self.zdim = cdim, zdim
        self.encoder = MLP(idim + cdim, out_dim=2*zdim, hidden=(128,128))
        self.prior   = MLP(cdim, out_dim=2*zdim, hidden=(128,128))
        self.decoder = DiagGaussianHead(zdim + cdim, 2*idim)
        
    def attach_normalizers(self, scaler_r, scaler_v, scaler_rnext):
        """Attach normalization scalers for automatic preprocessing."""
        self.scaler_r = scaler_r
        self.scaler_v = scaler_v
        self.scaler_rnext = scaler_rnext

    def encode(self, r_next, c):
        q = self.encoder(torch.cat([r_next, c], dim=-1))
        q_mu, q_logv = q.split(self.zdim, dim=-1)
        return q_mu, q_logv

    def prior_params(self, c):
        p = self.prior(c)
        p_mu, p_logv = p.split(self.zdim, dim=-1)
        return p_mu, p_logv

    def decode(self, z, c):
        mu, log_sig = self.decoder(torch.cat([z, c], dim=-1))
        return mu, log_sig

    def forward(self, r_next, c):
        p_mu, p_logv = self.prior_params(c)
        q_mu, q_logv = self.encode(r_next, c)
        z = reparam(q_mu, q_logv)
        dec_out = self.decode(z, c)
        return dec_out, (q_mu, q_logv), (p_mu, p_logv)

    @torch.no_grad()
    def sample_torch(self, c, T=1.0):
        """
        Sampling from torch tensor, no (de)normalisation.
        """
        p_mu, p_logv = self.prior_params(c)
        z = reparam(p_mu, p_logv)  # sample from p(z|c)
        
        mu, log_sig = self.decode(z, c)
        r = mu + torch.exp(log_sig) * torch.randn_like(mu) * T
        return r
    
    @torch.no_grad()
    def sample(self, c_np, T=1.0, device=None):
        """
        Sample r_{n+1} in physical units given c = [r_n, v_n] as NumPy array. 
        Built in normalisation of input and denormalisation of output.

        Args:
            r_n_np (np.ndarray): shape (..., 3)
            v_n_np (np.ndarray): shape (..., 3)
            T (float): temperature scaling factor for stochasticity
            device (torch.device): GPU/CPU device to use (optional)

        Returns:
            np.ndarray: generated r_{n+1} in same physical scale as input
        """
        if device is None:
            device = next(self.parameters()).device

        # --- Normalize inputs ---
        v_n_np, r_n_np = c_np[...,:3], c_np[..., 3:]
        r_norm = self.scaler_r.transform(r_n_np)
        v_norm = self.scaler_v.transform(v_n_np)
        c = np.concatenate([r_norm, v_norm], axis=-1)

        # --- Convert to torch tensor ---
        c_t = torch.tensor(c, dtype=torch.float32, device=device)

        # --- Sample from conditional prior and decode ---
        p_mu, p_logv = self.prior_params(c_t)
        z = p_mu + torch.exp(0.5 * p_logv) * torch.randn_like(p_mu)
        mu_r, log_sig_r = self.decode(z, c_t)
        r_next_norm = mu_r + torch.exp(log_sig_r) * torch.randn_like(mu_r) * T

        # --- De-normalize to physical scale ---
        r_next_np = r_next_norm.cpu().numpy()
        r_next_phys = self.scaler_rnext.inverse_transform(r_next_np)

        return r_next_phys

In [72]:
def gaussian_nll_diag(x, mu, log_sig):
    # x, mu, log_sig: [..., 3]
    # Sum over dims; mean over batch
    return 0.5 * ((x - mu)**2 * torch.exp(-2*log_sig) + 2*log_sig + torch.log(torch.tensor(2*math.pi, device=x.device))).sum(-1).mean()

def kl_diag(q_mu, q_logv, p_mu, p_logv):
    # all [..., zdim]
    # KL(q||p) for diagonal Gaussians; mean over batch
    return 0.5 * (torch.exp(q_logv - p_logv) + (q_mu - p_mu)**2 * torch.exp(-p_logv) - 1 + p_logv - q_logv).sum(-1).mean()

def elbo_loss(r_next, dec_out, q, p, beta=1.0, full_cov=False):
    q_mu, q_logv = q
    p_mu, p_logv = p

    mu, log_sig = dec_out
    nll = gaussian_nll_diag(r_next, mu, log_sig)
    kl  = kl_diag(q_mu, q_logv, p_mu, p_logv)
    return nll + beta * kl, nll, kl

In [73]:
def train_cvae(model, train_loader, val_loader=None,
                epochs=50, lr=1e-3, beta=1.0, 
                grad_clip=1.0, save_path=None):

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.cuda.amp.GradScaler()  # mixed precision
    
    for epoch in range(1, epochs+1):
        model.train()
        total_loss, total_nll, total_kl = 0, 0, 0

        loop = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}")
        for batch in loop:
            # Batch: (r_n, v_n, r_{n+1})
            r_n, v_n, r_next = [x.to(device) for x in batch]
            c = torch.cat([r_n, v_n], dim=-1)  # conditioning variable
            
            optimizer.zero_grad()
            with torch.cuda.amp.autocast():
                dec_out, q, p = model(r_next, c)
                q_mu, q_logv = q
                p_mu, p_logv = p
                mu_r, log_sig_r = dec_out

                nll = gaussian_nll_diag(r_next, mu_r, log_sig_r)
                kl  = kl_diag(q_mu, q_logv, p_mu, p_logv)
                loss = nll + beta * kl

            scaler.scale(loss).backward()
            if grad_clip is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
            
            total_loss += loss.item()
            total_nll += nll.item()
            total_kl += kl.item()

            loop.set_postfix(loss=loss.item(), NLL=nll.item(), KL=kl.item())

        scheduler.step()
        print(f"Epoch {epoch}: loss={total_loss/len(train_loader):.4f}, "
              f"NLL={total_nll/len(train_loader):.4f}, KL={total_kl/len(train_loader):.4f}")

        # ---- optional validation ----
        if val_loader is not None and epoch % 5 == 0:
            val_loss = evaluate_cvae(model, val_loader, beta)
            print(f"Validation loss: {val_loss:.4f}")

        if save_path:
            torch.save({'model_state': model.state_dict()}, save_path)

In [75]:
@torch.no_grad()
def evaluate_cvae(model, loader, beta=1.0):
    model.eval()
    total = 0
    for batch in loader:
        r_n, v_n, r_next = [x.to(device) for x in batch]
        c = torch.cat([r_n, v_n], dim=-1)
        dec_out, q, p = model(r_next, c)
        q_mu, q_logv = q
        p_mu, p_logv = p
        mu_r, log_sig_r = dec_out
        nll = gaussian_nll_diag(r_next, mu_r, log_sig_r)
        kl  = kl_diag(q_mu, q_logv, p_mu, p_logv)
        total += (nll + beta*kl).item()
    return total / len(loader)

In [76]:
class RVDataset(Dataset):
    def __init__(self, r, v, r_next):
        self.r, self.v, self.r_next = r, v, r_next
    def __len__(self):
        return len(self.r)
    def __getitem__(self, idx):
        return self.r[idx], self.v[idx], self.r_next[idx]

In [77]:
# System type: 'bistable', 'dimer'
systemType = 'bistable'

# Conditioning variables: piri, piririm, pipimri, etc. - for dimer, piridqi
conditionedOn = 'piri'

# datapoint = [time (1), qi (3), vi (3), ? (1), ri(3)] -- 11 dim
# for dimer, alternating between particle 1 and particle 2.

# Datasets directory
localDirectory = "/group/ag_cmb/scratch/maojrs/stochasticClosure/" + systemType + "/boxsize5/benchmark/"

# Total no. of datasets
n_datasets = 50
train_split = 0.8

# Sample simulation files randomly
fnums = np.random.choice(2500, n_datasets, replace=False)
print(np.sort(fnums))
dataset = None

for f_num in fnums:
    try:
        ds = torch.Tensor(trajectoryTools.loadTrajectory(localDirectory + "simMoriZwanzig_", f_num)).unsqueeze(0)
    except FileNotFoundError:
        print(f'File {f_num} not available.')
        continue
              
    if dataset is None:
        dataset = ds
    else:
        dataset = torch.cat((dataset, ds), dim=0)
        
n_timesteps = dataset.shape[1]

# Dataset - training data
dataset.shape, dataset.flatten(end_dim=1).shape, n_timesteps, dataset.dtype

[  50   53   68  100  178  251  299  306  425  436  454  462  495  515
  527  618  702  706  712  833  935 1006 1111 1162 1196 1218 1410 1482
 1572 1649 1692 1765 1827 1880 2016 2050 2131 2151 2238 2268 2311 2322
 2332 2351 2352 2361 2393 2420 2459 2482]


(torch.Size([50, 10000, 11]), torch.Size([500000, 11]), 10000, torch.float32)

In [78]:
q = dataset[..., 1:4]   # position (not used now, but may be later)
v = dataset[..., 4:7]   # velocity
r = dataset[..., 8:11]  # auxiliary var

In [79]:
r_n = r[:, :-1, :].flatten(end_dim=1)
v_n = v[:, :-1, :].flatten(end_dim=1)
r_next = r[:, 1:, :].flatten(end_dim=1)

In [80]:
## Normalization

scaler_r = StandardScaler().fit(r_n)
scaler_v = StandardScaler().fit(v_n)
scaler_rnext = StandardScaler().fit(r_next)

r_n_norm = torch.tensor(scaler_r.transform(r_n), dtype=torch.float32)
v_n_norm = torch.tensor(scaler_v.transform(v_n), dtype=torch.float32)
r_next_norm = torch.tensor(scaler_rnext.transform(r_next), dtype=torch.float32)

In [81]:
r_n.mean(), r_n.std(), v_n.mean(), v_n.std(), r_next.mean(), r_next.std()

(tensor(3.2306e-05),
 tensor(0.0162),
 tensor(-0.0002),
 tensor(0.1429),
 tensor(3.2519e-05),
 tensor(0.0162))

In [84]:
## Saving Scaler Params
joblib.dump(
    {
        "scaler_r": scaler_r,
        "scaler_v": scaler_v,
        "scaler_rnext": scaler_rnext
    },
    "normalizers.pkl"
)

['normalizers.pkl']

In [63]:
N = len(r_n_norm)
split = int(0.8*N)

In [64]:
train_ds = RVDataset(r_n_norm[:split], v_n_norm[:split], r_next_norm[:split])
val_ds   = RVDataset(r_n_norm[split:], v_n_norm[split:], r_next_norm[split:])

In [67]:
train_loader = DataLoader(train_ds, batch_size=4096, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=4096, shuffle=False, num_workers=4)

model = CVAE()

In [70]:
train_cvae(model,
           train_loader,
           val_loader=val_loader,
           epochs=50,
           lr=1e-3,
           beta=1.0,
           grad_clip=1.0,
           save_path="cvae_checkpoint.pt")

/srv/data/jakut77/miniconda3/envs/pyta/lib/python3.12/site-packages/torch/amp/grad_scaler.py:131: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(
Epoch 1/50:   0%|                                                                                                  | 0/98 [00:00<?, ?it/s]/srv/data/jakut77/miniconda3/envs/pyta/lib/python3.12/site-packages/torch/amp/autocast_mode.py:250: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
Epoch 1/50: 100%|██████████████████████████████████████████████████████████| 98/98 [00:12<00:00,  7.71it/s, KL=0.398, NLL=1.02, loss=1.42]


Epoch 1: loss=1.8272, NLL=1.4850, KL=0.3422


Epoch 2/50: 100%|█████████████████████████████████████████████████████████| 98/98 [00:12<00:00,  7.54it/s, KL=0.416, NLL=0.862, loss=1.28]


Epoch 2: loss=1.3266, NLL=0.9152, KL=0.4114


Epoch 3/50: 100%|█████████████████████████████████████████████████████████| 98/98 [00:12<00:00,  7.77it/s, KL=0.429, NLL=0.807, loss=1.24]


...

Epoch 49/50: 100%|████████████████████████████████████████████████████████| 98/98 [00:11<00:00,  8.51it/s, KL=0.511, NLL=0.636, loss=1.15]


Epoch 49: loss=1.1225, NLL=0.6120, KL=0.5105


Epoch 50/50: 100%|█████████████████████████████████████████████████████████| 98/98 [00:11<00:00,  8.25it/s, KL=0.503, NLL=0.55, loss=1.05]

Epoch 50: loss=1.1234, NLL=0.6127, KL=0.5107


Validation loss: 1.1305
